In [1]:
import os
import sys
sys.path.insert(0, "/home/thomasb/")

import numpy as np
import time
import matplotlib.pyplot as plt
import numba as nb
from scipy.stats import median_abs_deviation
from scipy import linalg
import rfitools
import importlib
from scipy.ndimage import median_filter
import os
from datetime import datetime
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import matplotlib.colors as mcolors
from albatros_analysis.src.correlations import timing_solution_class as tsc
import map_utils as mutils

importlib.reload(rfitools)

<module 'rfitools' from '/home/thomasb/albatros_analysis/scripts/mapmaking/rfitools.py'>

In [41]:
# VERSION LATE JULY
# fname="/scratch/thomasb/mohan/visibilities_mars_20250722" #/vis_1753200150:1753286400_bit=1_ant=7_pol=2_cha=360:392_tim=160652_upx=64_acc=512_ipfb=0.2_complex64_regular_20251024T003558.npy
# start_specnum = 1273601
# tstart = 1753200150
# nant = 7
# npol = 2
# chanstart, chanend = 360, 392
# acclen = 512
# osamp = 64

# VERSION 10.08
# fname="/scratch/thomasb/mohan/vis_ant=7_pol=2_cha=1834:1852_jul22_orbcomm_IQ_20260808T125039"
# start_specnum = 1273601
# tstart = 1753200150
# nant = 7
# npol = 2
# chanstart, chanend = 1834, 1852
# acclen = 512
# osamp = 64

# VERSION 20.08
fname = "/scratch/thomasb/mohan/vis_ant=6_pol=2_cha=1834:1852_pipeline_test_20260820T162649"
start_specnum = 3526673
tstart = 1762099360
nant = 6
npol = 2
chanstart, chanend = 1834, 1852
acclen = 1024
osamp = 4

In [42]:
T_SPECTRA = 4096/250e6
dt = acclen * osamp * T_SPECTRA
print('dt', dt)
df = 1/(osamp*T_SPECTRA)
print('df', df)

triu_idx=np.triu_indices(nant,k=1)
nbl = len(triu_idx[0])
print("num ant", nant, "num baselines", nbl)

freqs = 250e6 - (np.arange(chanstart*osamp,chanend*osamp)/osamp * 250e6/4096) # aliased
#freqs = np.arange(chanstart*osamp,chanend*osamp)/osamp * 250e6/4096          # NOT aliased
print('total number of frequencies', freqs.shape)
print('first frequency (MHz)', freqs[0]/1e6)
print('last frequency (MHz)', freqs[-1]/1e6)

dt 0.067108864
df 15258.789062499998
num ant 6 num baselines 15
total number of frequencies (72,)
first frequency (MHz) 138.0615234375
last frequency (MHz) 136.9781494140625


In [43]:
# load in all the data
arr = rfitools.load_all_parts(fname)

Found 149 files. Pre-allocating and loading...
shape of first file (21, 8668, 72, 4)
  Processed vis_1762099360:1762185620_bit=1_ant=6_pol=2_cha=1834:1852_tim=1285360_upx=4_acc=1024_ipfb=0.4_complex64_pipeline_test_20260820T162649_part00001.npy into slice [17336:26004]
  Processed vis_1762099360:1762185620_bit=1_ant=6_pol=2_cha=1834:1852_tim=1285360_upx=4_acc=1024_ipfb=0.4_complex64_pipeline_test_20260820T162649_part00002.npy into slice [26004:34672]
  Processed vis_1762099360:1762185620_bit=1_ant=6_pol=2_cha=1834:1852_tim=1285360_upx=4_acc=1024_ipfb=0.4_complex64_pipeline_test_20260820T162649_part00003.npy into slice [34672:43340]
  Processed vis_1762099360:1762185620_bit=1_ant=6_pol=2_cha=1834:1852_tim=1285360_upx=4_acc=1024_ipfb=0.4_complex64_pipeline_test_20260820T162649_part00004.npy into slice [43340:52008]
  Processed vis_1762099360:1762185620_bit=1_ant=6_pol=2_cha=1834:1852_tim=1285360_upx=4_acc=1024_ipfb=0.4_complex64_pipeline_test_20260820T162649_part00005.npy into slice [520

In [46]:
ntimes, nchans, _ = arr.shape
print('ntimes', ntimes)
print('nchans', nchans)
print('nbl', nbl)

ntimes 1284732
nchans 72
nbl 15


In [47]:
# set up timing solution object
tsobj = tsc.TimingSolution(tstart, '/scratch/thomasb/timing_solution')

In [48]:
tsobj.UTC_offset

1762099302.3440464

In [49]:
tsobj.UTC_per_spec

1.6383945659016475e-05

In [50]:
# get start time of first visibility via first baseband spectrum
unix_start = start_specnum * tsobj.UTC_per_spec + tsobj.UTC_offset
#unix_start = start_specnum * UTC_per_spec + UTC_offset
print('corrected unix start', unix_start)
# previously was something like 1753200149.2883923 1753286370.0452518

# make fine time array, placing times in center of visibility spectra
fine_tarr = unix_start + (np.arange(ntimes) + 0.5) * dt
print('fine time array shape', fine_tarr.shape)
print('last center time according to array', fine_tarr[-1])
print('expected (ish)', tstart + 86200)

corrected unix start 1762099360.124865
fine time array shape (1284732,)
last center time according to array 1762185576.996375
expected (ish) 1762185560


In [51]:
coords = {
        #0: [79.417161473, -90.767238685, 187.9577], #MARS1
        0: [79.417198047, -90.758739192, 183.0684], #MARS2
        1: [79.388456412, -91.019202963, 25.1938], #CSA
        2: [79.418302573, -90.667395452, 59.6242], #Mars 5
        3: [79.397984238, -90.799842408, 41.6994], #Mars 6
        4: [79.411474117, -90.695266129, 31.6314], #Mars 7
        5: [79.443757694, -90.718202634, 414.9131] #MARS8
    }
#antmap = {0:"MARS1", 1:"MARS2",2:"MARS4",3:"MARS5",4:"MARS6",5:"MARS7",6:"MARS8"}
antmap = {0:"MARS2", 1:"MARS4",2:"MARS5",3:"MARS6",4:"MARS7",5:"MARS8"}
ant0=EarthLocation.from_geodetic(lat=coords[0][0], lon=coords[0][1], height=coords[0][2])

In [52]:
arr.shape

(1284732, 72, 21)

# RFI Flagging

In [33]:
# do some RFI stuff
npmasks = np.zeros((arr.shape[0], arr.shape[1], nbl),dtype='bool',order='C')
kf=3
kt=3
kp=5
for i in range(nbl):
    ai = triu_idx[0][i]
    aj = triu_idx[1][i]
    # if ai==aj or ai==1 or aj==1: 
    #     continue #skip autos and MARS2
    vis = arr[:,:,i + ai + 1]
    vis_amp = np.abs(vis)
    med_spec = rfitools.fast_median(vis_amp,axis=0)
    # med_spec_test = np.median(vis_amp,axis=0)
    # assert np.array_equal(med_spec, med_spec_test)
    # print(med_spec.shape)
    #folded_med_spec = med_spec.reshape(-1,64)
    folded_med_spec = med_spec.reshape(-1,nchans)
    folded_med_spec_med = np.median(folded_med_spec[:-2,:],axis=0)
    # plt.plot(folded_med_spec_med)
    vis_amp_flat = vis_amp/np.tile(folded_med_spec_med,folded_med_spec.shape[0])
    time_window = int(600/dt) # 10 minute window
    med_time = rfitools.fast_median(vis_amp_flat,axis=1)
    # med_time_test = np.median(vis_amp_flat,axis=1)
    # assert np.array_equal(med_time, med_time_test)
    
    tp = median_filter(med_time,time_window)
    vis_amp_double_flat = vis_amp_flat/tp[:, None]
    time_mask , time_score = rfitools.time_mask(vis_amp_double_flat, kt)
    freq_mask , freq_score = rfitools.freq_mask(vis_amp_double_flat, kf)
    time_flagged_fraction = np.sum(time_mask)/len(time_mask)
    freq_flagged_fraction = np.sum(freq_mask)/len(freq_mask)
    # plt.clf()
    # fig=plt.gcf()
    # fig.set_size_inches(10,4)
    # plt.suptitle(f"{antmap[ai]}-{antmap[aj]}")
    # plt.subplot(121)
    # plt.title(f"time flag {time_flagged_fraction*100:.1f}%")
    # ax=plt.gca()
    # ax2=ax.twinx()
    # ax.plot(time_score)
    # ax2.plot(time_mask, ls='dotted',c='green')
    # plt.subplot(122)
    # plt.title(f"freq flag {freq_flagged_fraction*100:.1f}%")
    # ax=plt.gca()
    # ax2=ax.twinx()
    # ax.plot(freq_score)
    # ax2.plot(freq_mask, ls='dotted',c='green')
    # plt.tight_layout()
    # plt.savefig(os.path.join(imgdir, f"{antmap[ai]}-{antmap[aj]}_time_flagging.png"),dpi=300)
    sign_geo = -1
    sign_clock = -1
    mask = time_mask[:,None] | freq_mask[None,:]
    npmasks[:,:,i] = mask
    print("processed", i, antmap[ai],antmap[aj], f"time {time_flagged_fraction*100:.1f}% freq {freq_flagged_fraction*100:.1f}%")

/home/thomasb/.virtualenvs/tcenv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/thomasb/.virtualenvs/tcenv/lib/python3.11/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


processed 0 MARS2 MARS4 time 0.0% freq 0.0%
processed 1 MARS2 MARS5 time 0.0% freq 0.0%
processed 2 MARS2 MARS6 time 0.0% freq 0.0%
processed 3 MARS2 MARS7 time 0.0% freq 0.0%
processed 4 MARS2 MARS8 time 0.0% freq 0.0%
processed 5 MARS4 MARS5 time 0.0% freq 0.0%
processed 6 MARS4 MARS6 time 0.0% freq 0.0%
processed 7 MARS4 MARS7 time 0.0% freq 0.0%
processed 8 MARS4 MARS8 time 0.0% freq 0.0%
processed 9 MARS5 MARS6 time 0.0% freq 0.0%
processed 10 MARS5 MARS7 time 0.0% freq 0.0%
processed 11 MARS5 MARS8 time 0.0% freq 0.0%
processed 12 MARS6 MARS7 time 0.0% freq 0.0%
processed 13 MARS6 MARS8 time 0.0% freq 0.0%
processed 14 MARS7 MARS8 time 0.0% freq 0.0%


In [53]:
print('masks shape', npmasks.shape)
print('data shape', arr.shape)

masks shape (1284732, 72, 15)
data shape (1284732, 72, 21)


# Determine Clock Delays and Satellite Passes

In [54]:
# (optional) mohan thing
# osamp=64
# dt = 512 * 4096*osamp/250e6
# start_specnum = 1273601
# UTC_offset = 1753200128.4140258
# UTC_per_spec = 1.638402806904218e-05
# unix_start = start_specnum * UTC_per_spec + UTC_offset
# ntimes = 160573
# fine_tarr = unix_start + (np.arange(ntimes) + 0.5) * dt

In [55]:
# find out when the sat is up
sat_up = tsobj.get_data_mask(fine_tarr, 2)

# Find where the mask changes
changes = np.diff(sat_up.astype(int))

starts = np.where(changes == 1)[0] + 1
ends = np.where(changes == -1)[0] + 1

# Handle runs at the beginning/end
if sat_up[0]:
    starts = np.r_[0, starts]
if sat_up[-1]:
    ends = np.r_[ends, len(sat_up)]

# List of slices
satpasses = [slice(s, e) for s, e in zip(starts, ends)]
print(len(satpasses))
print(satpasses)

18
[slice(np.int64(76247), np.int64(78211), None), slice(np.int64(123419), np.int64(125382), None), slice(np.int64(165629), np.int64(167592), None), slice(np.int64(214874), np.int64(216838), None), slice(np.int64(255339), np.int64(257302), None), slice(np.int64(303181), np.int64(305144), None), slice(np.int64(345746), np.int64(347710), None), slice(np.int64(527392), np.int64(529356), None), slice(np.int64(581406), np.int64(583370), None), slice(np.int64(614809), np.int64(616773), None), slice(np.int64(672097), np.int64(674060), None), slice(np.int64(704920), np.int64(706883), None), slice(np.int64(763568), np.int64(765532), None), slice(np.int64(853327), np.int64(855291), None), slice(np.int64(1034871), np.int64(1036834), None), slice(np.int64(1125702), np.int64(1127665), None), slice(np.int64(1214842), np.int64(1216806), None), slice(np.int64(1253553), np.int64(1255516), None)]


In [56]:
s = satpasses[0]
print(len(range(*s.indices(ntimes))))

1964


In [39]:
for s in satpasses:
    st = fine_tarr[s][0]
    end = fine_tarr[s][-1]
    print(st, end, end-st)

1762104477.007973 1762104608.742673 131.73469996452332
1762107642.6673055 1762107774.3348966 131.6675910949707
1762110475.332455 1762110607.000046 131.6675910949707
1762113780.1084626 1762113911.8431625 131.73469996452332
1762116495.6686444 1762116627.3362355 131.6675910949707
1762119706.290916 1762119837.958507 131.6675910949707
1762122562.779712 1762122694.5144122 131.7347002029419
1762134752.8364222 1762134884.5711222 131.73469996452332
1762138377.6546023 1762138509.3893023 131.73469996452332
1762140619.2919865 1762140751.0266864 131.73469996452332
1762144463.8245873 1762144595.4921784 131.6675910949707
1762146666.5388303 1762146798.2064216 131.66759133338928
1762150602.3394861 1762150734.0741863 131.7347002029419
1762156625.96401 1762156757.69871 131.73469996452332
1762168809.175616 1762168940.8432071 131.6675910949707
1762174904.740842 1762175036.4084332 131.6675910949707
1762180886.824979 1762181018.559679 131.73469996452332
1762183484.6762133 1762183616.3438044 131.6675910949707

In [58]:
# save all the pulses
p = '/scratch/thomasb/data_without_clock/pipeline_test_phases2'
#p = '/scratch/thomasb/testing_extracting_satpasses'
os.makedirs(p, exist_ok= True)

for pidx, satpass in enumerate(satpasses):

    # get the Unix time array corresponding to the visibilities around the pulse
    unix_pass = fine_tarr[satpass]
    print('Unix times around satellite pass', unix_pass.shape)
    print('Duration of satellite pulse (s)', unix_pass[-1]-unix_pass[0])

    # get the clock delays for each baseline at the visibility times
    clock_delays_pass = tsobj.interpolate_delay2(unix_pass, extrapolate=True)
    clock_delays_pass, b = tsobj.all_blines(clock_delays_pass)
    clock_delays_pass *= 1e-9 #turn into ns
    print('clock delays shape', clock_delays_pass.shape)

    # cut the visibility to that region
    vis_pass = np.zeros((len(unix_pass), nchans, nbl), dtype=arr.dtype)
    for blidx in range(nbl):
        ai = triu_idx[0][blidx]
        vis_pass[:, :, blidx] = arr[satpass, :, blidx+ai+1]

    # correct for clock
    # vis_pass *= np.exp(
    #     2j*np.pi
    #     *freqs[None, :, None] # (BD, nchans, BD)
    #     *clock_delays_pass.T[:, None, :] # (ntimes, BD, nbl)
    #     )

    # get the mask for the pass unix times
    mask_pass = np.transpose(npmasks[:, satpass, :], (1, 2, 0))

    print('pass visibility shape', vis_pass.shape)
    print('pass mask shape', mask_pass.shape)
    print('pass times shape', unix_pass.shape)
    print('pass frequencies shape', vis_pass.shape)
    print('all frequencies shape', freqs.shape)
    
    np.savez(
        os.path.join(p, f'satpass_{pidx}.npz'),
        data = vis_pass,
        mask = mask_pass,
        times = unix_pass,
        freqs = freqs,
    )

Unix times around satellite pass (1964,)
Duration of satellite pulse (s) 131.73469996452332
number of baselines (containing ref ant) in tau data: 5
number of desired interpolation unix times: 1964
number of total data unix times (2160,)
clock delays shape (15, 1964)
pass visibility shape (1964, 72, 15)
pass mask shape (0, 15, 1284732)
pass times shape (1964,)
pass frequencies shape (1964, 72, 15)
all frequencies shape (72,)
Unix times around satellite pass (1963,)
Duration of satellite pulse (s) 131.6675910949707
number of baselines (containing ref ant) in tau data: 5
number of desired interpolation unix times: 1963
number of total data unix times (2160,)
clock delays shape (15, 1963)
pass visibility shape (1963, 72, 15)
pass mask shape (0, 15, 1284732)
pass times shape (1963,)
pass frequencies shape (1963, 72, 15)
all frequencies shape (72,)
Unix times around satellite pass (1963,)
Duration of satellite pulse (s) 131.6675910949707
number of baselines (containing ref ant) in tau data: 